In [1]:
import os
import bisect
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

TRAIN_DIR = "../data/processed/phase2/train"
VAL_DIR   = "../data/processed/phase2/val"
TEST_DIR  = "../data/processed/phase2/test"


class EEGWindowDataset(Dataset):
    def __init__(self, split_dir):
        self.split_dir = split_dir
        self.file_records = []   # each item: dict with x_path, y_path, n_samples
        self.cumulative_sizes = []
        self.total_samples = 0

        x_files = sorted([f for f in os.listdir(split_dir) if f.endswith("_X.npy")])

        for x_file in x_files:
            y_file = x_file.replace("_X.npy", "_y.npy")
            x_path = os.path.join(split_dir, x_file)
            y_path = os.path.join(split_dir, y_file)

            if not os.path.exists(y_path):
                print(f"Skipping {x_file} because matching y file is missing")
                continue

            X = np.load(x_path, mmap_mode="r")
            y = np.load(y_path, mmap_mode="r")

            if len(X) != len(y):
                print(f"Skipping {x_file} due to mismatch: X={X.shape}, y={y.shape}")
                continue

            n_samples = len(X)

            self.file_records.append({
                "x_path": x_path,
                "y_path": y_path,
                "n_samples": n_samples,
                "file_name": x_file
            })

            self.total_samples += n_samples
            self.cumulative_sizes.append(self.total_samples)

        print(f"\nIndexed split: {split_dir}")
        print(f"Files: {len(self.file_records)}")
        print(f"Total samples: {self.total_samples}")

    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx):
        if idx < 0 or idx >= self.total_samples:
            raise IndexError("Index out of range")

        file_idx = bisect.bisect_right(self.cumulative_sizes, idx)
        prev_cum = 0 if file_idx == 0 else self.cumulative_sizes[file_idx - 1]
        local_idx = idx - prev_cum

        record = self.file_records[file_idx]

        X = np.load(record["x_path"], mmap_mode="r")
        y = np.load(record["y_path"], mmap_mode="r")

        x_item = X[local_idx]
        y_item = y[local_idx]

        x_item = torch.tensor(x_item, dtype=torch.float32)
        y_item = torch.tensor(y_item, dtype=torch.float32)

        return x_item, y_item

In [2]:
train_dataset = EEGWindowDataset(TRAIN_DIR)
val_dataset = EEGWindowDataset(VAL_DIR)
test_dataset = EEGWindowDataset(TEST_DIR)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

for batch_X, batch_y in train_loader:
    print("Batch X shape:", batch_X.shape)
    print("Batch y shape:", batch_y.shape)
    print("First batch labels:", batch_y[:20])
    break



Indexed split: ../data/processed/phase2/train
Files: 175
Total samples: 445471

Indexed split: ../data/processed/phase2/val
Files: 21
Total samples: 37197

Indexed split: ../data/processed/phase2/test
Files: 42
Total samples: 72911
Batch X shape: torch.Size([64, 18, 1280])
Batch y shape: torch.Size([64])
First batch labels: tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])


In [3]:
print(len(train_dataset), len(val_dataset), len(test_dataset))

445471 37197 72911


step 2 - class imbalance check and initializing BCE with logit loss

In [4]:
import numpy as np
import os

def compute_class_counts(split_dir):
    x_files = sorted([f for f in os.listdir(split_dir) if f.endswith("_X.npy")])
    
    total_pos = 0
    total_neg = 0

    for x_file in x_files:
        y_file = x_file.replace("_X.npy", "_y.npy")
        y_path = os.path.join(split_dir, y_file)

        if not os.path.exists(y_path):
            continue

        y = np.load(y_path, mmap_mode="r")

        pos = np.sum(y == 1)
        neg = np.sum(y == 0)

        total_pos += pos
        total_neg += neg

    return total_pos, total_neg

In [5]:
TRAIN_DIR = "../data/processed/phase2/train"

num_pos, num_neg = compute_class_counts(TRAIN_DIR)

print("Positive (seizure):", num_pos)
print("Negative (non-seizure):", num_neg)

pos_weight = num_neg / num_pos

print("\nComputed pos_weight:", pos_weight)

Positive (seizure): 2104
Negative (non-seizure): 443367

Computed pos_weight: 210.72576045627378


In [6]:
import torch

pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32)

In [7]:
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

In [8]:
print("pos_weight:", pos_weight)

pos_weight: 210.72576045627378


step 3 - model - CNN and LSTM

In [9]:
import torch
import torch.nn as nn

class EEGCNNLSTM(nn.Module):
    def __init__(self):
        super().__init__()

        # -----------------------------------------
        # CNN block
        # Input: (batch, 18, 1280)
        # -----------------------------------------
        self.conv1 = nn.Conv1d(in_channels=18, out_channels=32, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm1d(32)
        self.pool1 = nn.MaxPool1d(kernel_size=2)   # 1280 -> 640
        self.dropout1 = nn.Dropout(0.3)

        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm1d(64)
        self.pool2 = nn.MaxPool1d(kernel_size=2)   # 640 -> 320
        self.dropout2 = nn.Dropout(0.3)

        # -----------------------------------------
        # LSTM block
        # After convs: (batch, 64, 320)
        # LSTM input:  (batch, 320, 64)
        # -----------------------------------------
        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=False
        )

        self.dropout_lstm = nn.Dropout(0.3)

        # -----------------------------------------
        # Classifier
        # -----------------------------------------
        self.fc1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        # x: (batch, 18, 1280)

        x = self.conv1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.pool1(x)
        x = self.dropout1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = torch.relu(x)
        x = self.pool2(x)
        x = self.dropout2(x)

        # x: (batch, 64, 320)
        x = x.transpose(1, 2)   # -> (batch, 320, 64)

        lstm_out, (hidden, cell) = self.lstm(x)

        # hidden shape: (num_layers, batch, hidden_size)
        x = hidden[-1]   # (batch, 64)

        x = self.dropout_lstm(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        # output shape: (batch, 1)
        return x.squeeze(1)   # -> (batch,)

In [10]:
model = EEGCNNLSTM()

for batch_X, batch_y in train_loader:
    outputs = model(batch_X)
    print("Input shape:", batch_X.shape)
    print("Output shape:", outputs.shape)
    print("Label shape:", batch_y.shape)
    break

Input shape: torch.Size([64, 18, 1280])
Output shape: torch.Size([64])
Label shape: torch.Size([64])


In [11]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

Using device: mps


In [12]:
model = EEGCNNLSTM().to(device)
pos_weight_tensor = pos_weight_tensor.to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4) # decreasing the learning due to irregular jumps in gradients and exponential learning

In [13]:
batch_X = batch_X.to(device)
batch_y = batch_y.to(device)

In [14]:
for batch_X, batch_y in train_loader:
    batch_X = batch_X.to(device)
    out = model(batch_X)
    print(out.shape)
    break

torch.Size([64])


step 4 - training loop

In [15]:
import torch
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

In [16]:


def train_one_epoch(model, loader, criterion, optimizer, device, max_batches=None):
    model.train()
    running_loss = 0.0
    total_samples_seen = 0

    for i, (batch_X, batch_y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_X.size(0)
        total_samples_seen += batch_X.size(0)

        if i % 100 == 0:
            print(f"Batch {i}/{len(loader)} | Loss: {loss.item():.4f}")

    epoch_loss = running_loss / total_samples_seen
    return epoch_loss

def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0

    all_labels = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_X)                 # raw logits
            loss = criterion(outputs, batch_y)

            probs = torch.sigmoid(outputs)           # convert logits -> probabilities
            preds = (probs >= 0.5).float()           # default threshold for now

            running_loss += loss.item() * batch_X.size(0)

            all_labels.extend(batch_y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    recall = recall_score(all_labels, all_preds, zero_division=0)
    precision = precision_score(all_labels, all_preds, zero_division=0)
    f1 = f1_score(all_labels, all_preds, zero_division=0)

    try:
        roc_auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        roc_auc = 0.0

    return epoch_loss, recall, precision, f1, roc_auc

In [17]:
import os
os.makedirs("../models", exist_ok=True)

In [18]:
num_epochs = 10

best_val_recall = 0.0
best_model_path = "../models/best_eeg_cnnlstm.pth"

train_losses = []
val_losses = []
val_recalls = []
val_precisions = []
val_f1s = []
val_aucs = []

for epoch in range(num_epochs):

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)

    val_loss, val_recall, val_precision, val_f1, val_auc = evaluate(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_recalls.append(val_recall)
    val_precisions.append(val_precision)
    val_f1s.append(val_f1)
    val_aucs.append(val_auc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss   : {train_loss:.4f}")
    print(f"  Val Loss     : {val_loss:.4f}")
    print(f"  Val Recall   : {val_recall:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val F1       : {val_f1:.4f}")
    print(f"  Val ROC-AUC  : {val_auc:.4f}")

    if val_recall > best_val_recall:
        best_val_recall = val_recall
        torch.save(model.state_dict(), best_model_path)
        print(f"  Saved best model to: {best_model_path}")

    print("-" * 50)

print("Training complete.")
print("Best validation recall:", best_val_recall)

Batch 0/6961 | Loss: 0.6759
Batch 100/6961 | Loss: 0.6263
Batch 200/6961 | Loss: 3.2318
Batch 300/6961 | Loss: 5.1121
Batch 400/6961 | Loss: 0.6313
Batch 500/6961 | Loss: 0.6408
Batch 600/6961 | Loss: 0.6795
Batch 700/6961 | Loss: 0.6706
Batch 800/6961 | Loss: 0.7205
Batch 900/6961 | Loss: 0.5308
Batch 1000/6961 | Loss: 0.6186
Batch 1100/6961 | Loss: 4.6099
Batch 1200/6961 | Loss: 0.7366
Batch 1300/6961 | Loss: 0.5430
Batch 1400/6961 | Loss: 0.7949
Batch 1500/6961 | Loss: 1.2558
Batch 1600/6961 | Loss: 0.5985
Batch 1700/6961 | Loss: 0.5974
Batch 1800/6961 | Loss: 0.2577
Batch 1900/6961 | Loss: 0.5635
Batch 2000/6961 | Loss: 0.2383
Batch 2100/6961 | Loss: 0.7264
Batch 2200/6961 | Loss: 0.5997
Batch 2300/6961 | Loss: 1.0751
Batch 2400/6961 | Loss: 0.5783
Batch 2500/6961 | Loss: 0.2659
Batch 2600/6961 | Loss: 0.2457
Batch 2700/6961 | Loss: 0.2669
Batch 2800/6961 | Loss: 1.1741
Batch 2900/6961 | Loss: 0.3604
Batch 3000/6961 | Loss: 0.3226
Batch 3100/6961 | Loss: 0.3477
Batch 3200/6961 | Lo


Shifted from feature-based models to deep learning using raw EEG windows (18 × 1280) as input.

Couldn’t load full dataset into memory, so built a lazy-loading PyTorch Dataset using mmap to avoid crashes.

Created DataLoaders that fetch data batch-wise instead of preloading everything.

Computed severe class imbalance (~210:1) and used it as pos_weight in BCEWithLogitsLoss.

This ensured the model does not collapse to predicting only “no seizure.”

Built a CNN + LSTM model:

CNN → captures local waveform/spike patterns

LSTM → captures temporal evolution across the window

Ensured correct tensor flow: (batch, 18, 1280) → CNN → transpose → LSTM → dense → logit.

Configured training on Apple MPS (since no CUDA), keeping batch size moderate.

Implemented training loop with batch-level logging to monitor progress early.

Ran a short training (limited batches) to validate behavior instead of full training.

Observed key result:

Recall improved significantly (~0.60 → model detects seizures now)

Precision extremely low (~0.01 → too many false positives)

Conclusion: model is no longer broken, imbalance handling works, but predictions are poorly calibrated and need tuning (threshold, learning rate, training stability).

In [ ]:
num_epochs = 2

best_val_recall = 0.0
best_model_path = "../models/best_eeg_cnnlstm.pth"

train_losses = []
val_losses = []
val_recalls = []
val_precisions = []
val_f1s = []
val_aucs = []

for epoch in range(num_epochs):

    train_loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
    max_batches=200)
    val_loss, val_recall, val_precision, val_f1, val_auc = evaluate(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_recalls.append(val_recall)
    val_precisions.append(val_precision)
    val_f1s.append(val_f1)
    val_aucs.append(val_auc)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss   : {train_loss:.4f}")
    print(f"  Val Loss     : {val_loss:.4f}")
    print(f"  Val Recall   : {val_recall:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val F1       : {val_f1:.4f}")
    print(f"  Val ROC-AUC  : {val_auc:.4f}")

    if val_recall > best_val_recall:
        best_val_recall = val_recall
        torch.save(model.state_dict(), best_model_path)
        print(f"  Saved best model to: {best_model_path}")

    print("-" * 50)

print("Training complete.")
print("Best validation recall:", best_val_recall)

	•	Built the Phase 4 pipeline on raw EEG windows instead of hand-engineered features.
	•	Used window input shape (18 channels, 1280 time steps) directly for deep learning.
	•	Since the dataset was too large to load into memory, switched to a lazy-loading PyTorch Dataset.
	•	Used mmap-based loading so files are read only when samples are needed.
	•	This avoided RAM crashes and made full training possible on the Mac.
	•	Created DataLoaders for train, validation, and test without merging all .npy files into one huge tensor.
	•	Computed the class imbalance from the training split and found a very high positive weight, around 210.7.
	•	Used this in BCEWithLogitsLoss(pos_weight=...) so the model does not ignore seizure windows.
	•	Built a CNN + LSTM model to capture both local waveform patterns and temporal dependencies.
	•	CNN layers were used first to reduce and encode signal patterns across time.
	•	LSTM was then used on top of CNN features to model sequence behavior within each EEG window.
	•	Since training was on Mac, used MPS/Metal instead of CUDA.
	•	First verified the full pipeline by checking tensor shapes and making sure the forward pass produced the right output shape.
	•	Added batch-level logging during training to monitor speed and early learning behavior.
	•	A short capped run initially showed high recall quickly, but that result turned out to be optimistic and not fully reliable.
	•	Full-data training gave a more realistic picture of model behavior.
	•	In full training, the model started learning slowly: train loss decreased from about 1.01 to 0.83 in the first two epochs.
	•	Validation recall also improved from about 0.04 to 0.11 in those first two full epochs.
	•	This confirmed that the model was not collapsed, but it was still weak and undertrained.
	•	Precision remained extremely low, around 0.01, meaning the model was producing many false positives.
	•	Validation ROC-AUC stayed around 0.44–0.46 early on, showing that class separation was still poor.
	•	Across later epochs, validation recall did not increase smoothly and instead fluctuated noticeably.
	•	For example, some epochs had recall around 0.09–0.20, while a later epoch jumped close to 0.6.
	•	At the same time, ROC-AUC stayed somewhat more stable, around 0.5–0.6, suggesting the ranking signal was present but weak.
	•	Main interpretation: the model is learning some seizure signal, but predictions are still unstable and threshold-sensitive.
	•	Because of this, the best saved checkpoint is more important than the final epoch checkpoint.
	•	Current conclusion: Phase 4 is working technically, the deep model is learning, but it is still not calibrated or robust enough to be considered reliable yet.

In [19]:
model.load_state_dict(torch.load("../models/best_eeg_cnnlstm.pth"))
model.eval()

EEGCNNLSTM(
  (conv1): Conv1d(18, 32, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool1): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.3, inplace=False)
  (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout2): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(64, 64, batch_first=True)
  (dropout_lstm): Dropout(p=0.3, inplace=False)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)

In [20]:
all_labels = []
all_probs = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)

        outputs = model(batch_X)
        probs = torch.sigmoid(outputs)

        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(batch_y.numpy())

In [21]:
from sklearn.metrics import recall_score, precision_score, f1_score

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

print("\nThreshold Evaluation:\n")

for t in thresholds:
    preds = (np.array(all_probs) >= t).astype(int)

    recall = recall_score(all_labels, preds, zero_division=0)
    precision = precision_score(all_labels, preds, zero_division=0)
    f1 = f1_score(all_labels, preds, zero_division=0)

    print(f"Threshold: {t}")
    print(f"  Recall   : {recall:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  F1       : {f1:.4f}")
    print("-" * 30)


Threshold Evaluation:

Threshold: 0.1
  Recall   : 1.0000
  Precision: 0.0050
  F1       : 0.0100
------------------------------
Threshold: 0.2
  Recall   : 1.0000
  Precision: 0.0060
  F1       : 0.0120
------------------------------
Threshold: 0.3
  Recall   : 1.0000
  Precision: 0.0071
  F1       : 0.0141
------------------------------
Threshold: 0.4
  Recall   : 1.0000
  Precision: 0.0098
  F1       : 0.0195
------------------------------
Threshold: 0.5
  Recall   : 1.0000
  Precision: 0.0148
  F1       : 0.0292
------------------------------
Threshold: 0.6
  Recall   : 0.9872
  Precision: 0.0205
  F1       : 0.0402
------------------------------
Threshold: 0.7
  Recall   : 0.9701
  Precision: 0.0285
  F1       : 0.0554
------------------------------
